<a href="https://colab.research.google.com/github/Arjita15/AI_LAB/blob/main/lab8/lab8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Title: Bahdanau Attention (Additive Attention)

#### Objectives
* To build an RNN Encoder–Decoder model with Bahdanau (Additive) Attention for English–French translation using PyTorch.
* To learn how Bahdanau Attention helps the decoder focus on the most important words in the input sentence during translation.
* To train the model using teacher forcing, the Adam optimizer, and Negative Log Likelihood (NLL) loss to improve translation performance.


#### Theory
This project implements an RNN Encoder–Decoder model with Bahdanau (Additive) Attention for translating English and French sentences using PyTorch. The encoder first changes each input word into a numerical embedding and processes the sentence through an RNN to create hidden state representations. Instead of using only the last hidden state from the encoder, the attention mechanism allows the decoder to look at all encoder hidden states and choose the most useful information at each step of translation. This helps the model produce better translations, especially for longer sentences. During training, teacher forcing is used, which means the correct target word is given as the next input to the decoder rather than the word predicted by the model. This makes training faster and more stable. The model is trained with the Adam optimizer, and the Negative Log Likelihood (NLL) loss function is used to calculate the prediction error and update the model's weights.

In [29]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


In [30]:
SOS_token = 0 # Start of the Sentence
EOS_token = 1 # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [31]:
# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [32]:
def readLangs(path:str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")

    # Read the file and split into lines
    lines = open(path, encoding='utf-8').\
        read().strip().split('\n')

    # Split every line into pairs and normalize (english to french)
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]

    # Reverse pairs: English-French -> French-English
    pairs = [list(reversed(p)) for p in pairs]

    # Input is French, output is English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)

    return input_lang, output_lang, pairs

In [33]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return len(p[0].split(' ')) < MAX_LENGTH and \
        len(p[1].split(' ')) < MAX_LENGTH and \
        p[1].startswith(eng_prefixes)


def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

In [34]:
def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [35]:
PATH = r'eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

output_lang.word2index['am']  # try different English words.

Reading lines...
Read 45690 sentence pairs
Trimmed to 3150 sentence pairs
Counting words...
Counted words:
fra 1706
eng 931
['vous etes vraiment embetants', 'you re really annoying']


15

In [36]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size

        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

In [37]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)
        self.Ua = nn.Linear(hidden_size, hidden_size)
        self.Va = nn.Linear(hidden_size, 1)

    def forward(self, query, keys):
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)

        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, keys)

        return context, weights

In [38]:
class AttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(AttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.rnn = nn.RNN(2 * hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []
        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions
    def forward_step(self, input, hidden, encoder_outputs):
        embedded =  self.dropout(self.embedding(input))

        query = hidden.permute(1, 0, 2) # seq_len, batch, hidden_size -> batch, seq_len, hidden_size
        context, attn_weights = self.attention(query, encoder_outputs)
        input_rnn = torch.cat((embedded, context), dim=2)

        output, hidden = self.rnn(input_rnn, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

In [39]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def tensorsFromPair(pair):
    input_tensor = tensorFromSentence(input_lang, pair[0])
    target_tensor = tensorFromSentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids
        train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))

    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

In [40]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
          decoder_optimizer, criterion):

    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor) # using teacher forcing

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [41]:
import time
import math

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

In [42]:
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker
import numpy as np

def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    # this locator puts ticks at regular intervals
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)

In [43]:
def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
               print_every=100, plot_every=100):
    start = time.time()
    plot_losses = []
    print_loss_total = 0  # Reset every print_every
    plot_loss_total = 0  # Reset every plot_every

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                        epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses)

In [44]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [45]:
def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

In [46]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)

# Put decoder creation here
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

Reading lines...
Read 45690 sentence pairs
Trimmed to 3150 sentence pairs
Counting words...
Counted words:
fra 1706
eng 931


In [47]:
train(
    train_dataloader,
    encoder,
    decoder,
    EPOCHS,
    print_every=5,
    plot_every=5
)

0m 19s (- 12m 58s) (5 2%) 1.8315
0m 33s (- 10m 30s) (10 5%) 1.0543
0m 46s (- 9m 30s) (15 7%) 0.7089
0m 59s (- 8m 57s) (20 10%) 0.4522
1m 13s (- 8m 32s) (25 12%) 0.2839
1m 26s (- 8m 10s) (30 15%) 0.1828
1m 39s (- 7m 50s) (35 17%) 0.1255
1m 52s (- 7m 31s) (40 20%) 0.0951
2m 6s (- 7m 14s) (45 22%) 0.0768
2m 19s (- 6m 58s) (50 25%) 0.0652
2m 32s (- 6m 43s) (55 27%) 0.0609
2m 46s (- 6m 27s) (60 30%) 0.0556
2m 59s (- 6m 12s) (65 32%) 0.0524
3m 12s (- 5m 57s) (70 35%) 0.0500
3m 26s (- 5m 43s) (75 37%) 0.0486
3m 39s (- 5m 29s) (80 40%) 0.0459
3m 52s (- 5m 15s) (85 42%) 0.0450
4m 6s (- 5m 1s) (90 45%) 0.0456
4m 19s (- 4m 47s) (95 47%) 0.0429
4m 32s (- 4m 32s) (100 50%) 0.0440
4m 46s (- 4m 19s) (105 52%) 0.0429
4m 59s (- 4m 5s) (110 55%) 0.0403
5m 13s (- 3m 51s) (115 57%) 0.0455
5m 26s (- 3m 37s) (120 60%) 0.0409
5m 40s (- 3m 24s) (125 62%) 0.0386
5m 53s (- 3m 10s) (130 65%) 0.0391
6m 6s (- 2m 56s) (135 67%) 0.0390
6m 19s (- 2m 42s) (140 70%) 0.0412
6m 32s (- 2m 29s) (145 72%) 0.0413
6m 45s (- 2

In [48]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder)

> nous fetons ca
= we re celebrating
< we re celebrating <EOS>

> ils sont tres bienveillants
= they are very kind
< they are very kind <EOS>

> je suis curieux
= i am curious
< i m curious <EOS>

> nous abandonnons
= we re quitting
< we re giving up <EOS>

> vous etes toutes pretes
= you re all set
< you re all set <EOS>

> nous sommes impuissantes
= we re powerless
< we re helpless <EOS>

> nous sommes armes
= we re armed
< we re armed <EOS>

> tu es contrariee
= you re upset
< you re upset <EOS>

> nous avons toutes peur
= we re all afraid
< we re all afraid <EOS>

> nous sommes remues
= we re shaken
< we re shaken <EOS>



#### Discussion

The model demonstrates an attention-based RNN Encoder-Decoder for English-French machine translation. The encoder processes the French input, creating hidden state representations for each word. The decoder then generates the English translation word by word. Unlike models that rely solely on the encoder's final hidden state, Bahdanau Attention enables the decoder to prioritize relevant parts of the input sentence at each decoding step. This improves contextual understanding and translation quality, especially for longer sentences. The dataset undergoes preprocessing, including text cleaning, sentence filtering, vocabulary construction, numerical conversion of words, and the addition of SOS and EOS tokens. The model is trained iteratively with mini-batches, and the consistent decrease in loss indicates effective learning of the French-English language mapping. Ultimately, the attention mechanism significantly enhances translation accuracy by optimizing the utilization of input sentence information.

#### Conclusion

The Bahdanau Attention-based RNN Encoder-Decoder model proves to be an effective approach for machine translation. Its attention mechanism enables the decoder to focus on the most relevant input words, moving beyond the limitations of a single context vector. This results in more accurate and coherent English translations from French sentences. This project underscores the vital role of attention in sequence-to-sequence learning and offers a foundational understanding for more advanced neural translation architectures such as Transformers.